In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')
print("Training data shape:", train_data.shape)
print("Test data shape:", test_data.shape)
print("\nFirst few rows:")
train_data.head()

In [ ]:
print("Dataset Info:")
print(train_data.info())
print("\nStatistical Summary:")
train_data.describe()

In [ ]:
transported_counts = train_data['Transported'].value_counts()
print("\nTarget Variable Distribution:")
print(transported_counts)
print(f"\nPercentage Transported: {transported_counts[True] / len(train_data) * 100:.2f}%")
plt.figure(figsize=(8, 6))
transported_counts.plot(kind='bar', color=['skyblue', 'salmon'])
plt.xlabel('Transported')
plt.ylabel('Count')
plt.title('Distribution of Target Variable')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
missing_values = train_data.isnull().sum()
missing_percent = 100 * missing_values / len(train_data)
missing_table = pd.concat([missing_values, missing_percent], axis=1)
missing_table.columns = ['Missing Values', 'Percentage']
missing_table = missing_table[missing_table['Missing Values'] > 0].sort_values('Percentage', ascending=False)
print("Missing Values Analysis:")
print(missing_table)
if len(missing_table) > 0:
    plt.figure(figsize=(10, 6))
    missing_table['Percentage'].plot(kind='barh')
    plt.xlabel('Percentage of Missing Values')
    plt.title('Features with Missing Values')
    plt.tight_layout()
    plt.show()

In [ ]:
categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
for idx, col in enumerate(categorical_cols):
    if col in train_data.columns:
        transported_by_feature = train_data.groupby(col)['Transported'].value_counts(normalize=True).unstack()
        transported_by_feature.plot(kind='bar', ax=axes[idx], stacked=False)
        axes[idx].set_title(f'Transported by {col}')
        axes[idx].set_ylabel('Proportion')
        axes[idx].legend(title='Transported')
        axes[idx].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()
for idx, col in enumerate(numerical_cols):
    if col in train_data.columns:
        train_data.boxplot(column=col, by='Transported', ax=axes[idx])
        axes[idx].set_title(f'{col} by Transported Status')
        axes[idx].set_xlabel('Transported')
plt.suptitle('')  
plt.tight_layout()
plt.show()

In [ ]:
numeric_data = train_data[numerical_cols + ['Transported']].copy()
numeric_data['Transported'] = numeric_data['Transported'].astype(int)
plt.figure(figsize=(10, 8))
correlation_matrix = numeric_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()
print("\nCorrelation with Transported:")
print(correlation_matrix['Transported'].sort_values(ascending=False))

In [ ]:
y_train = train_data['Transported'].copy()
train_ids = train_data['PassengerId']
test_ids = test_data['PassengerId']
train_data = train_data.drop(['PassengerId', 'Transported'], axis=1)
test_data = test_data.drop(['PassengerId'], axis=1)
all_data = pd.concat([train_data, test_data], axis=0, ignore_index=True)
print(f"Combined data shape: {all_data.shape}")

In [ ]:
if 'Cabin' in all_data.columns:
    all_data['Cabin_Deck'] = all_data['Cabin'].str.split('/').str[0]
    all_data['Cabin_Num'] = all_data['Cabin'].str.split('/').str[1]
    all_data['Cabin_Side'] = all_data['Cabin'].str.split('/').str[2]
    all_data = all_data.drop('Cabin', axis=1)
    print("Created Cabin features: Deck, Num, Side")
if 'Name' in all_data.columns:
    all_data = all_data.drop('Name', axis=1)

In [ ]:
spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
all_data['TotalSpending'] = all_data[spending_cols].sum(axis=1)
all_data['HasSpending'] = (all_data['TotalSpending'] > 0).astype(int)
print("Created spending features: TotalSpending, HasSpending")

In [ ]:
categorical_features = all_data.select_dtypes(include=['object', 'bool']).columns
for col in categorical_features:
    all_data[col].fillna(all_data[col].mode()[0] if not all_data[col].mode().empty else 'Unknown', inplace=True)
numerical_features = all_data.select_dtypes(include=[np.number]).columns
for col in numerical_features:
    all_data[col].fillna(all_data[col].median(), inplace=True)
print(f"Missing values after imputation: {all_data.isnull().sum().sum()}")

In [ ]:
categorical_features = all_data.select_dtypes(include=['object', 'bool']).columns
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))
    label_encoders[col] = le
print(f"Encoded {len(categorical_features)} categorical features")
print(f"Final features: {list(all_data.columns)}")

In [ ]:
X_train = all_data[:len(train_data)]
X_test = all_data[len(train_data):]
print(f"Final training set shape: {X_train.shape}")
print(f"Final test set shape: {X_test.shape}")

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
print(f"Training set: {X_tr.shape}")
print(f"Validation set: {X_val.shape}")
print(f"\nClass distribution in training:")
print(y_tr.value_counts(normalize=True))

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_val, y_val, model_name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    print(f"\n{'='*60}")
    print(f"{model_name} Results:")
    print(f"{'='*60}")
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"\nClassification Report (Validation):")
    print(classification_report(y_val, y_val_pred))
    cm = confusion_matrix(y_val, y_val_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    if hasattr(model, 'predict_proba'):
        y_val_proba = model.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_val_proba)
        print(f"ROC AUC Score: {roc_auc:.4f}")
        fpr, tpr, _ = roc_curve(y_val, y_val_proba)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], 'k--', label='Random')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - {model_name}')
        plt.legend()
        plt.tight_layout()
        plt.show()
    return model, val_acc

In [ ]:
models = {}
results = {}
lr_model, lr_acc = evaluate_classifier(
    LogisticRegression(max_iter=1000, random_state=42),
    X_tr, y_tr, X_val, y_val,
    "Logistic Regression"
)
models['Logistic Regression'] = lr_model
results['Logistic Regression'] = lr_acc

In [ ]:
dt_model, dt_acc = evaluate_classifier(
    DecisionTreeClassifier(max_depth=10, random_state=42),
    X_tr, y_tr, X_val, y_val,
    "Decision Tree"
)
models['Decision Tree'] = dt_model
results['Decision Tree'] = dt_acc

In [ ]:
rf_model, rf_acc = evaluate_classifier(
    RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    X_tr, y_tr, X_val, y_val,
    "Random Forest"
)
models['Random Forest'] = rf_model
results['Random Forest'] = rf_acc

In [ ]:
gb_model, gb_acc = evaluate_classifier(
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    X_tr, y_tr, X_val, y_val,
    "Gradient Boosting"
)
models['Gradient Boosting'] = gb_model
results['Gradient Boosting'] = gb_acc

In [ ]:
results_df = pd.DataFrame(list(results.items()), columns=['Model', 'Validation Accuracy'])
results_df = results_df.sort_values('Validation Accuracy', ascending=False)
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(results_df.to_string(index=False))
plt.figure(figsize=(10, 6))
plt.barh(results_df['Model'], results_df['Validation Accuracy'])
plt.xlabel('Validation Accuracy')
plt.title('Model Comparison - Validation Accuracy')
plt.xlim(0.7, 0.85)
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    print(f"\nTop 15 Important Features ({best_model_name}):")
    print(feature_importance.head(15))
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance.head(15)['Feature'], feature_importance.head(15)['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Top 15 Feature Importances ({best_model_name})')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
print(f"Performing hyperparameter tuning for {best_model_name}...\n")
if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 15, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
elif best_model_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7]
    }
    grid_search = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
else:
    grid_search = None
    print("No hyperparameter tuning defined for this model.")
if grid_search:
    grid_search.fit(X_tr, y_tr)
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    tuned_model = grid_search.best_estimator_
    y_val_pred = tuned_model.predict(X_val)
    tuned_acc = accuracy_score(y_val, y_val_pred)
    print(f"Tuned model validation accuracy: {tuned_acc:.4f}")
    if tuned_acc > results[best_model_name]:
        best_model = tuned_model
        print(f"\nUsing tuned model (improvement: {tuned_acc - results[best_model_name]:.4f})")

In [ ]:
print(f"Training {best_model_name} on full training data...")
best_model.fit(X_train, y_train)
test_predictions = best_model.predict(X_test)
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': test_predictions
})
submission.to_csv('submission.csv', index=False)
print("\nSubmission file created: submission.csv")
print(f"\nPrediction Distribution:")
print(submission['Transported'].value_counts())
print(f"\nPercentage Transported: {submission['Transported'].sum() / len(submission) * 100:.2f}%")